In [ ]:
pip install requests yt-dlp pydub tqdm

In [ ]:
import os
import json
import time
import requests
from pathlib import Path
from tqdm import tqdm

# ─── API KEY ───────────────────────────────────────────
FREESOUND_API_KEY = "B5HDbaf0EMUzw0NMHKI9IcPqtyyq6AEitL2v3Cxa"
# ───────────────────────────────────────────────────────────

OUTPUT_DIR = Path("dataset")
MAX_POR_CLASSE = 50
DURACAO_MAX_SEG = 10

CLASSES = {
    "socorro": [
        "woman screaming help",
        "female scream danger",
        "woman crying help",
    ],
    "grito_medo": [
        "woman scream fear",
        "female scream horror",
        "scream panic woman",
    ],
    "choro_distress": [
        "woman crying distress",
        "female sobbing panic",
    ],
    "ambiente_neutro": [
        "street ambient noise",
        "crowd noise outdoor",
        "rain background noise",
    ],
    "voz_normal": [
        "woman talking normal",
        "female voice conversation",
    ],
}

FREESOUND_BASE = "https://freesound.org/apiv2"

def buscar_sons(query, num=15):
    url = f"{FREESOUND_BASE}/search/text/"
    params = {
        "query": query,
        "fields": "id,name,duration,previews,license",
        "filter": f"duration:[0 TO {DURACAO_MAX_SEG}]",
        "page_size": num,
        "token": FREESOUND_API_KEY,
    }
    try:
        resp = requests.get(url, params=params, timeout=10)
        resp.raise_for_status()
        return resp.json().get("results", [])
    except Exception as e:
        print(f"  [ERRO] {query}: {e}")
        return []

def baixar_audio(som, destino):
    preview_url = som.get("previews", {}).get("preview-hq-mp3") or \
                  som.get("previews", {}).get("preview-lq-mp3")
    if not preview_url:
        return False
    nome = destino / f"fs_{som['id']}.mp3"
    if nome.exists():
        return True
    try:
        r = requests.get(preview_url, timeout=15)
        r.raise_for_status()
        nome.write_bytes(r.content)
        return True
    except:
        return False

print("Iniciando coleta...\n")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
total_geral = 0

for classe, queries in CLASSES.items():
    pasta = OUTPUT_DIR / classe
    pasta.mkdir(parents=True, exist_ok=True)
    coletados = 0
    print("Coletando classe: {classe}")

    for query in queries:
        if coletados >= MAX_POR_CLASSE:
            break
        resultados = buscar_sons(query)
        for som in resultados:
            if coletados >= MAX_POR_CLASSE:
                break
            if baixar_audio(som, pasta):
                coletados += 1
            time.sleep(0.3)

    total_geral += coletados
    print(f"   ✅ {coletados} áudios salvos\n")

print(f"🎉 Coleta finalizada! Total: {total_geral} arquivos")
print(f"📂 Pasta: {OUTPUT_DIR.resolve()}")

In [ ]:
import os
from pathlib import Path
dataset = Path("dataset")
for classe in sorted(dataset.iterdir()):
     arquivos = list(classe.glob("*.mp3"))
print(f"  {classe.name:<20} {len(arquivos):>4} arquivos")


In [ ]:
# AUGMENTATION

from pydub import AudioSegment
from pathlib import Path
import random, os
from tqdm import tqdm

OUTPUT_DIR = Path("dataset")
AUG_DIR = Path("dataset_augmented")

#originais + gera variações
def augmentar(audio, nome_base, destino):
    arquivos_gerados = []

    # 1. Original (cópia)
    p = destino / f"{nome_base}_original.wav"
    audio.export(str(p), format="wav")
    arquivos_gerados.append(p)

    # 2. Mais alto (simular grito próximo)
    p = destino / f"{nome_base}_alto.wav"
    (audio + 6).export(str(p), format="wav")
    arquivos_gerados.append(p)

    # 3. Mais baixo (simular distância)
    p = destino / f"{nome_base}_baixo.wav"
    (audio - 8).export(str(p), format="wav")
    arquivos_gerados.append(p)

    # 4. Mais lento (pitch levemente alterado)
    lento = audio._spawn(audio.raw_data, overrides={
        "frame_rate": int(audio.frame_rate * 0.88)
    }).set_frame_rate(audio.frame_rate)
    p = destino / f"{nome_base}_lento.wav"
    lento.export(str(p), format="wav")
    arquivos_gerados.append(p)

    # 5. Mais rápido
    rapido = audio._spawn(audio.raw_data, overrides={
        "frame_rate": int(audio.frame_rate * 1.12)
    }).set_frame_rate(audio.frame_rate)
    p = destino / f"{nome_base}_rapido.wav"
    rapido.export(str(p), format="wav")
    arquivos_gerados.append(p)

    # 6. Com ruído de fundo leve
    ruido = AudioSegment.silent(duration=len(audio)).overlay(
        AudioSegment.silent(duration=len(audio)) - 30
    )
    p = destino / f"{nome_base}_ruido.wav"
    audio.overlay(ruido).export(str(p), format="wav")
    arquivos_gerados.append(p)

    return arquivos_gerados

print("Iniciando data augmentation...\n")
resumo = {}

for pasta_classe in sorted(OUTPUT_DIR.iterdir()):
    if not pasta_classe.is_dir():
        continue

    classe = pasta_classe.name
    destino = AUG_DIR / classe
    destino.mkdir(parents=True, exist_ok=True)

    arquivos = list(pasta_classe.glob("*.mp3")) + list(pasta_classe.glob("*.wav"))
    total_gerado = 0

    print(f"📁 {classe} ({len(arquivos)} originais)")

    for arquivo in tqdm(arquivos, desc=f"   Processando"):
        try:
            audio = AudioSegment.from_file(str(arquivo))
            gerados = augmentar(audio, arquivo.stem, destino)
            total_gerado += len(gerados)
        except Exception as e:
            print(f"   [AVISO] {arquivo.name}: {e}")

    resumo[classe] = total_gerado
    print(f"   ✅ {total_gerado} arquivos gerados\n")

print("=" * 40)
print("📊 RESUMO FINAL")
print("=" * 40)
total = 0
for classe, qtd in resumo.items():
    print(f"  {classe:<20} {qtd:>4} arquivos")
    total += qtd
print(f"\n  TOTAL GERAL: {total} arquivos")
print(f"  📂 Pasta: {AUG_DIR.resolve()}")

In [ ]:
!pip install librosa scikit-learn numpy matplotlib seaborn

In [ ]:
import librosa
import numpy as np
from pathlib import Path
from tqdm import tqdm
import pickle

DATASET_DIR = Path("dataset_augmented")
FEATURES_FILE = "features.pkl"
N_MFCC = 40          # número de coeficientes MFCC
MAX_PAD = 174        # padronizar tamanho dos vetores

def extrair_mfcc(caminho_audio):
    """Extrai features MFCC de um arquivo de áudio."""
    try:
        audio, sr = librosa.load(str(caminho_audio), sr=22050, duration=5.0)
        mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=N_MFCC)

        # Padronizar tamanho
        if mfcc.shape[1] < MAX_PAD:
            pad = MAX_PAD - mfcc.shape[1]
            mfcc = np.pad(mfcc, ((0, 0), (0, pad)), mode="constant")
        else:
            mfcc = mfcc[:, :MAX_PAD]

        return mfcc.flatten()  # vetor 1D: 40 × 174 = 6960 valores
    except Exception as e:
        return None

print("Extraindo features MFCC...\n")

X = []  # features
y = []  # labels (classes)
classes = sorted([p.name for p in DATASET_DIR.iterdir() if p.is_dir()])
class_map = {nome: idx for idx, nome in enumerate(classes)}

print(f"Classes encontradas: {classes}\n")

for classe in classes:
    pasta = DATASET_DIR / classe
    arquivos = list(pasta.glob("*.wav")) + list(pasta.glob("*.mp3"))
    print(f"📁 {classe} — {len(arquivos)} arquivos")

    for arquivo in tqdm(arquivos, desc=f"   {classe}"):
        features = extrair_mfcc(arquivo)
        if features is not None:
            X.append(features)
            y.append(class_map[classe])

X = np.array(X)
y = np.array(y)

# Salvar features
with open(FEATURES_FILE, "wb") as f:
    pickle.dump({"X": X, "y": y, "classes": classes}, f)

print(f"\n✅ Features extraídas!")
print(f"   Shape X: {X.shape}")
print(f"   Shape y: {y.shape}")
print(f"   Salvo em: {FEATURES_FILE}")

In [ ]:
import pickle
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, confusion_matrix
import joblib

# Carregar features
with open("features.pkl", "rb") as f:
    data = pickle.load(f)

X, y, classes = data["X"], data["y"], data["classes"]

print("Dataset carregado: {X.shape[0]} amostras, {len(classes)} classes\n")

# Dividir treino/teste
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Normalizar
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Treinar modelo (Rede Neural MLP)
print("Treinando modelo...\n")
modelo = MLPClassifier(
    hidden_layer_sizes=(256, 128, 64),
    activation="relu",
    max_iter=300,
    random_state=42,
    verbose=True,
    early_stopping=True,
    validation_fraction=0.1,
)
modelo.fit(X_train, y_train)

# Avaliar
print("AVALIAÇÃO DO MODELO")
print("=" * 50)
y_pred = modelo.predict(X_test)
print(classification_report(y_test, y_pred, target_names=classes))

# Matriz de confusão
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=classes, yticklabels=classes)
plt.title("Matriz de Confusão")
plt.ylabel("Real")
plt.xlabel("Previsto")
plt.tight_layout()
plt.savefig("matriz_confusao.png", dpi=150)
plt.show()
print("Matriz salva em: matriz_confusao.png")

# Salvar modelo
joblib.dump(modelo, "modelo_deteccao.pkl")
joblib.dump(scaler, "scaler.pkl")
print("Modelo salvo em: modelo_deteccao.pkl")
print("Scaler salvo em: scaler.pkl")


In [ ]:
# CÉLULA NOVA

import requests
import time
from pathlib import Path
from tqdm import tqdm

FREESOUND_API_KEY = "B5HDbaf0EMUzw0NMHKI9IcPqtyyq6AEitL2v3Cxa"
FREESOUND_BASE = "https://freesound.org/apiv2"
OUTPUT_DIR = Path("dataset")
DURACAO_MAX_SEG = 10
MAX_POR_QUERY = 25

# Queries focadas nas classes fracas + português
QUERIES_EXTRAS = {
    "socorro": [
        "mulher gritando socorro",
        "woman scream help me",
        "female distress scream",
        "grito feminino desespero",
        "woman screaming attack",
        "female panic scream loud",
        "woman yelling help",
        "scream woman street",
    ],
    "choro_distress": [
        "mulher chorando desespero",
        "woman crying desperately",
        "female sobbing distress loud",
        "woman wailing crying",
        "choro feminino angustia",
        "woman crying panic",
        "female cry fear",
        "sobbing woman intense",
    ],
}

def buscar_sons(query, num=25):
    url = f"{FREESOUND_BASE}/search/text/"
    params = {
        "query": query,
        "fields": "id,name,duration,previews,license",
        "filter": f"duration:[0 TO {DURACAO_MAX_SEG}]",
        "page_size": num,
        "token": FREESOUND_API_KEY,
    }
    try:
        resp = requests.get(url, params=params, timeout=10)
        resp.raise_for_status()
        return resp.json().get("results", [])
    except Exception as e:
        print(f"  [ERRO] {query}: {e}")
        return []

def baixar_audio(som, destino):
    preview_url = som.get("previews", {}).get("preview-hq-mp3") or \
                  som.get("previews", {}).get("preview-lq-mp3")
    if not preview_url:
        return False
    nome = destino / f"fs_{som['id']}.mp3"
    if nome.exists():
        return True  # já existe, pula
    try:
        r = requests.get(preview_url, timeout=15)
        r.raise_for_status()
        nome.write_bytes(r.content)
        return True
    except:
        return False

print("Coletando mais áudios para classes fracas...\n")
resumo = {}

for classe, queries in QUERIES_EXTRAS.items():
    pasta = OUTPUT_DIR / classe
    pasta.mkdir(parents=True, exist_ok=True)

    # Contar quantos já existem
    ja_existem = len(list(pasta.glob("*.mp3")) + list(pasta.glob("*.wav")))
    print(f"📁 {classe} — já tem {ja_existem} arquivos")

    novos = 0
    for query in queries:
        resultados = buscar_sons(query, num=MAX_POR_QUERY)
        for som in tqdm(resultados, desc=f"  '{query}'", leave=False):
            if baixar_audio(som, pasta):
                novos += 1
        time.sleep(0.3)

    total = len(list(pasta.glob("*.mp3")) + list(pasta.glob("*.wav")))
    resumo[classe] = {"antes": ja_existem, "novos": novos, "total": total}
    print(f"  ✅ +{novos} novos | Total agora: {total} arquivos\n")

print("=" * 45)
print("RESUMO")
print("=" * 45)
for classe, info in resumo.items():
    print(f"  {classe:<20} {info['antes']} → {info['total']} arquivos (+{info['novos']})")

print("\n✅ Pronto! Agora rode o script de augmentation novamente nas pastas atualizadas.")

In [ ]:
# Augmentation nas classes atualizadas

from pydub import AudioSegment
from pathlib import Path
from tqdm import tqdm

DATASET_DIR = Path("dataset")
AUG_DIR = Path("dataset_augmented")

# Apenas as classes que receberam novos áudios
CLASSES_ATUALIZAR = ["socorro", "choro_distress"]

def augmentar(audio, nome_base, destino):
    gerados = 0
    for sufixo, fn in [
        ("original", lambda a: a),
        ("alto",     lambda a: a + 6),
        ("baixo",    lambda a: a - 8),
        ("lento",    lambda a: a._spawn(a.raw_data, overrides={"frame_rate": int(a.frame_rate * 0.88)}).set_frame_rate(a.frame_rate)),
        ("rapido",   lambda a: a._spawn(a.raw_data, overrides={"frame_rate": int(a.frame_rate * 1.12)}).set_frame_rate(a.frame_rate)),
        ("ruido",    lambda a: a - 5),
    ]:
        p = destino / f"{nome_base}_{sufixo}.wav"
        if not p.exists():  # pula se já existe
            try:
                fn(audio).export(str(p), format="wav")
                gerados += 1
            except:
                pass
    return gerados

print("Augmentation nas classes atualizadas...\n")

for classe in CLASSES_ATUALIZAR:
    pasta_orig = DATASET_DIR / classe
    pasta_aug  = AUG_DIR / classe
    pasta_aug.mkdir(parents=True, exist_ok=True)

    arquivos = list(pasta_orig.glob("*.mp3")) + list(pasta_orig.glob("*.wav"))
    total_gerado = 0

    print(f"📁 {classe} ({len(arquivos)} originais)")
    for arquivo in tqdm(arquivos, desc=f"   Processando"):
        try:
            audio = AudioSegment.from_file(str(arquivo))
            total_gerado += augmentar(audio, arquivo.stem, pasta_aug)
        except Exception as e:
            print(f"   [AVISO] {arquivo.name}: {e}")

    total_final = len(list(pasta_aug.glob("*.wav")))
    print(f"   ✅ {total_gerado} novos arquivos gerados | Total na pasta: {total_final}\n")

# Resumo geral do dataset_augmented
print("=" * 45)
print("DATASET FINAL COMPLETO")
print("=" * 45)
total_geral = 0
for pasta in sorted(AUG_DIR.iterdir()):
    if pasta.is_dir():
        qtd = len(list(pasta.glob("*.wav")))
        total_geral += qtd
        print(f"  {pasta.name:<20} {qtd:>4} arquivos")
print(f"\n  TOTAL GERAL: {total_geral} arquivos")

In [ ]:
import shutil
from pathlib import Path

DATASET_ORIG = Path("dataset_augmented")
DATASET_NOVO = Path("dataset_v2")

# Mapeamento: classe antiga -> classe nova
MAPEAMENTO = {
    "socorro":        "perigo",
    "choro_distress": "perigo",
    "grito_medo":     "grito_medo",
    "ambiente_neutro":"ambiente_neutro",
    "voz_normal":     "voz_normal",
}

print("📦 Reorganizando dataset...\n")

for classe_orig, classe_nova in MAPEAMENTO.items():
    pasta_orig = DATASET_ORIG / classe_orig
    pasta_nova = DATASET_NOVO / classe_nova
    pasta_nova.mkdir(parents=True, exist_ok=True)

    arquivos = list(pasta_orig.glob("*.wav")) + list(pasta_orig.glob("*.mp3"))
    for arquivo in arquivos:
        destino = pasta_nova / f"{classe_orig}_{arquivo.name}"
        if not destino.exists():
            shutil.copy2(arquivo, destino)

    print(f"  {classe_orig:<20} → {classe_nova}")

print("\n📊 NOVO DATASET:")
total = 0
for pasta in sorted(DATASET_NOVO.iterdir()):
    if pasta.is_dir():
        qtd = len(list(pasta.glob("*.wav")) + list(pasta.glob("*.mp3")))
        total += qtd
        print(f"  {pasta.name:<20} {qtd:>4} arquivos")
print(f"\n  TOTAL: {total} arquivos")

In [ ]:
#Retreinar o modelo

import pickle
import librosa
import numpy as np
from pathlib import Path
from tqdm import tqdm
import pickle

DATASET_DIR = Path("dataset_v2")
FEATURES_FILE = "features_v3.pkl"
N_MFCC = 40
MAX_PAD = 174

def extrair_mfcc(caminho_audio):
    try:
        audio, sr = librosa.load(str(caminho_audio), sr=22050, duration=5.0)
        mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=N_MFCC)
        if mfcc.shape[1] < MAX_PAD:
            mfcc = np.pad(mfcc, ((0, 0), (0, MAX_PAD - mfcc.shape[1])), mode="constant")
        else:
            mfcc = mfcc[:, :MAX_PAD]
        return mfcc.flatten()
    except:
        return None

print("🔍 Extraindo features...\n")

X, y = [], []
classes = sorted([p.name for p in DATASET_DIR.iterdir() if p.is_dir()])
class_map = {nome: idx for idx, nome in enumerate(classes)}
print(f"Classes: {classes}\n")

for classe in classes:
    pasta = DATASET_DIR / classe
    arquivos = list(pasta.glob("*.wav")) + list(pasta.glob("*.mp3"))
    print(f"📁 {classe} — {len(arquivos)} arquivos")
    for arquivo in tqdm(arquivos, desc=f"   {classe}"):
        features = extrair_mfcc(arquivo)
        if features is not None:
            X.append(features)
            y.append(class_map[classe])

X = np.array(X)
y = np.array(y)

with open(FEATURES_FILE, "wb") as f:
    pickle.dump({"X": X, "y": y, "classes": classes}, f)

print(f"\n✅ Features extraídas! Shape: {X.shape}")


# ✅ CÉLULA 3 — Treinar modelo final
# Cole numa célula nova e rode DEPOIS da célula acima

import pickle
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, confusion_matrix
import joblib

with open("features_v3.pkl", "rb") as f:
    data = pickle.load(f)

X, y, classes = data["X"], data["y"], data["classes"]
print(f"📦 Dataset: {X.shape[0]} amostras | Classes: {classes}\n")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

print("🧠 Treinando modelo final...\n")
modelo = MLPClassifier(
    hidden_layer_sizes=(256, 128, 64),
    activation="relu",
    max_iter=300,
    random_state=42,
    verbose=True,
    early_stopping=True,
    validation_fraction=0.1,
)
modelo.fit(X_train, y_train)

print("\n📊 AVALIAÇÃO MODELO FINAL")
print("=" * 50)
y_pred = modelo.predict(X_test)
print(classification_report(y_test, y_pred, target_names=classes))

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=classes, yticklabels=classes)
plt.title("Matriz de Confusão — Modelo Final")
plt.ylabel("Real")
plt.xlabel("Previsto")
plt.tight_layout()
plt.savefig("matriz_confusao_final.png", dpi=150)
plt.show()

joblib.dump(modelo, "modelo_final.pkl")
joblib.dump(scaler, "scaler_final.pkl")
print("\n💾 Modelo salvo: modelo_final.pkl")
print("💾 Scaler salvo: scaler_final.pkl")

In [ ]:
import librosa
import numpy as np
import joblib
import pickle
from google.colab import files

# Carregar modelo
modelo = joblib.load("modelo_final.pkl")
scaler = joblib.load("scaler_final.pkl")
with open("features_v3.pkl", "rb") as f:
    classes = pickle.load(f)["classes"]

print("✅ Modelo carregado!")
print(f"   Classes: {classes}\n")

def prever_audio(caminho):
    try:
        audio, sr = librosa.load(caminho, sr=22050, duration=5.0)
        mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=40)
        if mfcc.shape[1] < 174:
            mfcc = np.pad(mfcc, ((0,0),(0, 174 - mfcc.shape[1])), mode="constant")
        else:
            mfcc = mfcc[:, :174]
        features = scaler.transform([mfcc.flatten()])
        pred = modelo.predict(features)[0]
        proba = modelo.predict_proba(features)[0]

        print(f"🎯 Resultado: {classes[pred].upper()}")
        print("\nProbabilidades:")
        for c, p in zip(classes, proba):
            barra = "█" * int(p * 40)
            print(f"  {c:<20} {p:.1%}  {barra}")
    except Exception as e:
        print(f"[ERRO] {e}")

# Fazer upload do áudio
print("📂 Selecione um arquivo de áudio (.wav ou .mp3):")
uploaded = files.upload()

for nome_arquivo in uploaded.keys():
    print(f"\n🔊 Analisando: {nome_arquivo}")
    prever_audio(nome_arquivo)

In [ ]:
import librosa
import soundfile as sf
import numpy as np
import joblib
import pickle
from google.colab import files

modelo = joblib.load("modelo_final.pkl")
scaler = joblib.load("scaler_final.pkl")
with open("features_v3.pkl", "rb") as f:
    classes = pickle.load(f)["classes"]

print("📂 Selecione seu arquivo .m4a:")
uploaded = files.upload()

for nome in uploaded.keys():
    print(f"\n🔄 Convertendo {nome} para .wav...")
    audio, sr = librosa.load(nome, sr=22050)
    wav_path = nome.replace(".m4a", ".wav")
    sf.write(wav_path, audio, sr)
    print(f"✅ Convertido: {wav_path}")

    print(f"\n🔊 Analisando...")
    mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=40)
    if mfcc.shape[1] < 174:
        mfcc = np.pad(mfcc, ((0,0),(0, 174 - mfcc.shape[1])), mode="constant")
    else:
        mfcc = mfcc[:, :174]
    features = scaler.transform([mfcc.flatten()])
    pred = modelo.predict(features)[0]
    proba = modelo.predict_proba(features)[0]

    print(f"🎯 Resultado: {classes[pred].upper()}")
    print("\nProbabilidades:")
    for c, p in zip(classes, proba):
        barra = "█" * int(p * 40)
        print(f"  {c:<20} {p:.1%}  {barra}")

In [ ]:
import shutil
from pathlib import Path

DATASET_ORIG = Path("dataset_augmented")
DATASET_BINARIO = Path("dataset_binario")

MAPEAMENTO = {
    "socorro":        "perigo",
    "choro_distress": "perigo",
    "grito_medo":     "perigo",
    "ambiente_neutro":"nao_perigo",
    "voz_normal":     "nao_perigo",
}

print("📦 Reorganizando para modelo binário...\n")

for classe_orig, classe_nova in MAPEAMENTO.items():
    pasta_orig = DATASET_ORIG / classe_orig
    pasta_nova = DATASET_BINARIO / classe_nova
    pasta_nova.mkdir(parents=True, exist_ok=True)

    arquivos = list(pasta_orig.glob("*.wav")) + list(pasta_orig.glob("*.mp3"))
    for arquivo in arquivos:
        destino = pasta_nova / f"{classe_orig}_{arquivo.name}"
        if not destino.exists():
            shutil.copy2(arquivo, destino)

    print(f"  {classe_orig:<20} → {classe_nova}")

print("\n📊 DATASET BINÁRIO:")
total = 0
for pasta in sorted(DATASET_BINARIO.iterdir()):
    if pasta.is_dir():
        qtd = len(list(pasta.glob("*.wav")) + list(pasta.glob("*.mp3")))
        total += qtd
        print(f"  {pasta.name:<15} {qtd:>4} arquivos")
print(f"\n  TOTAL: {total} arquivos")


In [ ]:
import librosa
import numpy as np
from pathlib import Path
from tqdm import tqdm
import pickle

DATASET_DIR = Path("dataset_binario")
FEATURES_FILE = "features_binario.pkl"
N_MFCC = 40
MAX_PAD = 174

def extrair_mfcc(caminho_audio):
    try:
        audio, sr = librosa.load(str(caminho_audio), sr=22050, duration=5.0)
        mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=N_MFCC)
        if mfcc.shape[1] < MAX_PAD:
            mfcc = np.pad(mfcc, ((0, 0), (0, MAX_PAD - mfcc.shape[1])), mode="constant")
        else:
            mfcc = mfcc[:, :MAX_PAD]
        return mfcc.flatten()
    except:
        return None

print("🔍 Extraindo features...\n")

X, y = [], []
classes = sorted([p.name for p in DATASET_DIR.iterdir() if p.is_dir()])
class_map = {nome: idx for idx, nome in enumerate(classes)}
print(f"Classes: {classes}\n")

for classe in classes:
    pasta = DATASET_DIR / classe
    arquivos = list(pasta.glob("*.wav")) + list(pasta.glob("*.mp3"))
    print(f"📁 {classe} — {len(arquivos)} arquivos")
    for arquivo in tqdm(arquivos, desc=f"   {classe}"):
        features = extrair_mfcc(arquivo)
        if features is not None:
            X.append(features)
            y.append(class_map[classe])

X = np.array(X)
y = np.array(y)

with open(FEATURES_FILE, "wb") as f:
    pickle.dump({"X": X, "y": y, "classes": classes}, f)

print(f"\n✅ Features extraídas! Shape: {X.shape}")


In [ ]:
import pickle
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, confusion_matrix
import joblib

with open("features_binario.pkl", "rb") as f:
    data = pickle.load(f)

X, y, classes = data["X"], data["y"], data["classes"]
print(f"📦 Dataset: {X.shape[0]} amostras | Classes: {classes}\n")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

print("🧠 Treinando modelo binário...\n")
modelo = MLPClassifier(
    hidden_layer_sizes=(256, 128, 64),
    activation="relu",
    max_iter=300,
    random_state=42,
    verbose=True,
    early_stopping=True,
    validation_fraction=0.1,
)
modelo.fit(X_train, y_train)

print("\n📊 AVALIAÇÃO MODELO BINÁRIO")
print("=" * 50)
y_pred = modelo.predict(X_test)
print(classification_report(y_test, y_pred, target_names=classes))

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=classes, yticklabels=classes)
plt.title("Matriz de Confusão — Modelo Binário")
plt.ylabel("Real")
plt.xlabel("Previsto")
plt.tight_layout()
plt.savefig("matriz_confusao_binario.png", dpi=150)
plt.show()

joblib.dump(modelo, "modelo_binario.pkl")
joblib.dump(scaler, "scaler_binario.pkl")
print("\n💾 Modelo salvo: modelo_binario.pkl")
print("💾 Scaler salvo: scaler_binario.pkl")


In [ ]:
import librosa, numpy as np, joblib, pickle
from google.colab import files

modelo = joblib.load("modelo_binario.pkl")
scaler = joblib.load("scaler_binario.pkl")
with open("features_binario.pkl", "rb") as f:
    classes = pickle.load(f)["classes"]

print("📂 Selecione um áudio para testar:")
uploaded = files.upload()

for nome in uploaded.keys():
    audio, sr = librosa.load(nome, sr=22050, duration=5.0)
    mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=40)
    if mfcc.shape[1] < 174:
        mfcc = np.pad(mfcc, ((0,0),(0, 174 - mfcc.shape[1])), mode="constant")
    else:
        mfcc = mfcc[:, :174]
    features = scaler.transform([mfcc.flatten()])
    pred = modelo.predict(features)[0]
    proba = modelo.predict_proba(features)[0]

    emoji = "🚨" if classes[pred] == "perigo" else "✅"
    print(f"\n{emoji} Resultado: {classes[pred].upper()}")
    print("\nProbabilidades:")
    for c, p in zip(classes, proba):
        barra = "█" * int(p * 40)
        print(f"  {c:<15} {p:.1%}  {barra}")

In [ ]:
# ✅ CÉLULA NOVA — Coletar sons de contexto de briga/agressão
# Cole numa célula nova e rode

import requests
import time
from pathlib import Path
from tqdm import tqdm

FREESOUND_API_KEY = "B5HDbaf0EMUzw0NMHKI9IcPqtyyq6AEitL2v3Cxa"
FREESOUND_BASE = "https://freesound.org/apiv2"
OUTPUT_DIR = Path("dataset_binario")
DURACAO_MAX_SEG = 10
MAX_POR_QUERY = 20

# Novos sons de contexto
QUERIES_CONTEXTO = {
    "perigo": [
        # Voz masculina alterada / ameaça
        "man shouting angry",
        "male voice aggressive",
        "man yelling threatening",
        "angry male argument",
        # Briga / confronto
        "argument fight sounds",
        "domestic fight audio",
        "couple arguing loud",
        "people fighting shouting",
        # Impacto físico
        "hitting impact sound",
        "punch impact sound",
        "body impact thud",
        # Combinação choro + briga
        "woman crying man shouting",
        "female crying argument background",
    ],
    "nao_perigo": [
        # Conversas normais para balancear
        "people talking indoors",
        "casual conversation room",
        "man woman talking calm",
        "indoor conversation background",
    ],
}

def buscar_sons(query, num=20):
    url = f"{FREESOUND_BASE}/search/text/"
    params = {
        "query": query,
        "fields": "id,name,duration,previews,license",
        "filter": f"duration:[0 TO {DURACAO_MAX_SEG}]",
        "page_size": num,
        "token": FREESOUND_API_KEY,
    }
    try:
        resp = requests.get(url, params=params, timeout=10)
        resp.raise_for_status()
        return resp.json().get("results", [])
    except Exception as e:
        print(f"  [ERRO] {query}: {e}")
        return []

def baixar_audio(som, destino):
    preview_url = som.get("previews", {}).get("preview-hq-mp3") or \
                  som.get("previews", {}).get("preview-lq-mp3")
    if not preview_url:
        return False
    nome = destino / f"ctx_{som['id']}.mp3"
    if nome.exists():
        return True
    try:
        r = requests.get(preview_url, timeout=15)
        r.raise_for_status()
        nome.write_bytes(r.content)
        return True
    except:
        return False

print("🚀 Coletando sons de contexto...\n")
resumo = {}

for classe, queries in QUERIES_CONTEXTO.items():
    pasta = OUTPUT_DIR / classe
    pasta.mkdir(parents=True, exist_ok=True)
    antes = len(list(pasta.glob("*.mp3")) + list(pasta.glob("*.wav")))
    novos = 0

    print(f"📁 {classe} — já tem {antes} arquivos")

    for query in queries:
        resultados = buscar_sons(query)
        for som in tqdm(resultados, desc=f"  '{query}'", leave=False):
            if baixar_audio(som, pasta):
                novos += 1
        time.sleep(0.3)

    total = len(list(pasta.glob("*.mp3")) + list(pasta.glob("*.wav")))
    resumo[classe] = {"antes": antes, "novos": novos, "total": total}
    print(f"  ✅ +{novos} novos | Total: {total}\n")

print("=" * 45)
print("📊 RESUMO")
print("=" * 45)
for classe, info in resumo.items():
    print(f"  {classe:<15} {info['antes']} → {info['total']} (+{info['novos']})")
print("\n✅ Pronto! Agora rode o augmentation e retreine o modelo.")

In [ ]:
# ✅ CÉLULA — Coletar mais sons para nao_perigo
# Cole numa célula nova e rode

import requests
import time
from pathlib import Path
from tqdm import tqdm

FREESOUND_API_KEY = "B5HDbaf0EMUzw0NMHKI9IcPqtyyq6AEitL2v3Cxa"
FREESOUND_BASE = "https://freesound.org/apiv2"
PASTA_DESTINO = Path("dataset_binario/nao_perigo")
PASTA_DESTINO.mkdir(parents=True, exist_ok=True)
DURACAO_MAX_SEG = 10

QUERIES = [
    "people talking indoors",
    "casual conversation",
    "man talking calm",
    "woman talking indoor",
    "office background noise",
    "restaurant background chatter",
    "TV background noise",
    "family talking dinner",
    "children playing background",
    "indoor ambient voices",
    "male female conversation",
    "quiet room ambient",
    "supermarket ambient",
    "coffee shop background",
    "classroom background noise",
]

def buscar_sons(query, num=20):
    url = f"{FREESOUND_BASE}/search/text/"
    params = {
        "query": query,
        "fields": "id,name,duration,previews,license",
        "filter": f"duration:[0 TO {DURACAO_MAX_SEG}]",
        "page_size": num,
        "token": FREESOUND_API_KEY,
    }
    try:
        resp = requests.get(url, params=params, timeout=10)
        resp.raise_for_status()
        return resp.json().get("results", [])
    except Exception as e:
        print(f"  [ERRO] {query}: {e}")
        return []

def baixar_audio(som, destino):
    preview_url = som.get("previews", {}).get("preview-hq-mp3") or \
                  som.get("previews", {}).get("preview-lq-mp3")
    if not preview_url:
        return False
    nome = destino / f"nao_{som['id']}.mp3"
    if nome.exists():
        return True
    try:
        r = requests.get(preview_url, timeout=15)
        r.raise_for_status()
        nome.write_bytes(r.content)
        return True
    except:
        return False

antes = len(list(PASTA_DESTINO.glob("*.mp3")) + list(PASTA_DESTINO.glob("*.wav")))
print(f"🚀 Coletando mais sons para nao_perigo...")
print(f"   Antes: {antes} arquivos\n")

novos = 0
for query in QUERIES:
    resultados = buscar_sons(query)
    for som in tqdm(resultados, desc=f"  '{query}'", leave=False):
        if baixar_audio(som, PASTA_DESTINO):
            novos += 1
    time.sleep(0.3)

total = len(list(PASTA_DESTINO.glob("*.mp3")) + list(PASTA_DESTINO.glob("*.wav")))
print(f"\n✅ +{novos} novos arquivos")
print(f"   Total nao_perigo: {total} arquivos")

# Comparar com perigo
perigo = len(list(Path("dataset_binario/perigo").glob("*.mp3")) + list(Path("dataset_binario/perigo").glob("*.wav")))
print(f"   Total perigo:     {perigo} arquivos")
diff = perigo - total
if diff > 50:
    print(f"\n⚠️  Ainda desbalanceado ({diff} a menos). Pode rodar de novo.")
else:
    print(f"\n✅ Dataset equilibrado! Pode rodar o augmentation e retreinar.")

In [ ]:
# ✅ CÉLULA — Novas queries para nao_perigo
# Cole numa célula nova e rode

import requests
import time
from pathlib import Path
from tqdm import tqdm

FREESOUND_API_KEY = "B5HDbaf0EMUzw0NMHKI9IcPqtyyq6AEitL2v3Cxa"
FREESOUND_BASE = "https://freesound.org/apiv2"
PASTA_DESTINO = Path("dataset_binario/nao_perigo")
PASTA_DESTINO.mkdir(parents=True, exist_ok=True)

# Queries completamente diferentes das anteriores
QUERIES = [
    "rain window indoor",
    "birds chirping morning",
    "keyboard typing office",
    "footsteps walking indoor",
    "door opening closing",
    "car driving interior",
    "wind outside gentle",
    "water running tap",
    "television news background",
    "crowd murmur indoor",
    "phone ringing office",
    "elevator ambient",
    "bus interior ambient",
    "library quiet ambient",
    "park ambient daytime",
    "market outdoor ambient",
    "laughing people group",
    "children school background",
    "music store background",
    "airport terminal ambient",
]

def buscar_sons(query, num=20):
    url = f"{FREESOUND_BASE}/search/text/"
    params = {
        "query": query,
        "fields": "id,name,duration,previews,license",
        "filter": "duration:[0 TO 10]",
        "page_size": num,
        "token": FREESOUND_API_KEY,
    }
    try:
        resp = requests.get(url, params=params, timeout=10)
        resp.raise_for_status()
        return resp.json().get("results", [])
    except Exception as e:
        print(f"  [ERRO] {query}: {e}")
        return []

def baixar_audio(som, destino):
    preview_url = som.get("previews", {}).get("preview-hq-mp3") or \
                  som.get("previews", {}).get("preview-lq-mp3")
    if not preview_url:
        return False
    nome = destino / f"nao2_{som['id']}.mp3"
    if nome.exists():
        return True
    try:
        r = requests.get(preview_url, timeout=15)
        r.raise_for_status()
        nome.write_bytes(r.content)
        return True
    except:
        return False

antes = len(list(PASTA_DESTINO.glob("*.mp3")) + list(PASTA_DESTINO.glob("*.wav")))
print(f"🚀 Coletando com novas queries...")
print(f"   Antes: {antes} arquivos\n")

novos = 0
for query in QUERIES:
    resultados = buscar_sons(query)
    for som in tqdm(resultados, desc=f"  '{query}'", leave=False):
        if baixar_audio(som, PASTA_DESTINO):
            novos += 1
    time.sleep(0.3)

total = len(list(PASTA_DESTINO.glob("*.mp3")) + list(PASTA_DESTINO.glob("*.wav")))
perigo = len(list(Path("dataset_binario/perigo").glob("*.mp3")) + list(Path("dataset_binario/perigo").glob("*.wav")))

print(f"\n✅ +{novos} novos arquivos")
print(f"   Total nao_perigo: {total} arquivos")
print(f"   Total perigo:     {perigo} arquivos")

diff = abs(perigo - total)
if diff > 50:
    print(f"\n⚠️  Ainda desbalanceado ({diff} de diferença).")
else:
    print(f"\n✅ Dataset equilibrado! Pode rodar o augmentation e retreinar.")

In [ ]:
# ✅ CÉLULA 1 — Augmentation apenas no nao_perigo para equilibrar
# Cole numa célula nova e rode

from pydub import AudioSegment
from pathlib import Path
from tqdm import tqdm

PASTA_NAO_PERIGO = Path("dataset_binario/nao_perigo")
PASTA_PERIGO = Path("dataset_binario/perigo")

total_perigo = len(list(PASTA_PERIGO.glob("*.mp3")) + list(PASTA_PERIGO.glob("*.wav")))
total_nao = len(list(PASTA_NAO_PERIGO.glob("*.mp3")) + list(PASTA_NAO_PERIGO.glob("*.wav")))
faltam = total_perigo - total_nao

print(f"📊 Situação atual:")
print(f"   perigo:     {total_perigo} arquivos")
print(f"   nao_perigo: {total_nao} arquivos")
print(f"   Faltam:     {faltam} arquivos no nao_perigo\n")

def augmentar_um(audio, nome_base, destino, sufixo):
    p = destino / f"aug_{sufixo}_{nome_base}.wav"
    if p.exists():
        return False
    try:
        if sufixo == "alto":
            resultado = audio + 5
        elif sufixo == "baixo":
            resultado = audio - 8
        elif sufixo == "lento":
            resultado = audio._spawn(audio.raw_data, overrides={
                "frame_rate": int(audio.frame_rate * 0.88)
            }).set_frame_rate(audio.frame_rate)
        elif sufixo == "rapido":
            resultado = audio._spawn(audio.raw_data, overrides={
                "frame_rate": int(audio.frame_rate * 1.12)
            }).set_frame_rate(audio.frame_rate)
        else:
            resultado = audio
        resultado.export(str(p), format="wav")
        return True
    except:
        return False

arquivos = list(PASTA_NAO_PERIGO.glob("*.mp3")) + list(PASTA_NAO_PERIGO.glob("*.wav"))
gerados = 0
sufixos = ["alto", "baixo", "lento", "rapido"]
idx = 0

print(f"🔄 Gerando {faltam} arquivos extras para nao_perigo...\n")

while gerados < faltam:
    arquivo = arquivos[idx % len(arquivos)]
    sufixo = sufixos[(idx // len(arquivos)) % len(sufixos)]
    try:
        audio = AudioSegment.from_file(str(arquivo))
        if augmentar_um(audio, arquivo.stem, PASTA_NAO_PERIGO, f"{sufixo}_{idx}"):
            gerados += 1
    except:
        pass
    idx += 1
    if idx > len(arquivos) * 10:
        break

total_final = len(list(PASTA_NAO_PERIGO.glob("*.mp3")) + list(PASTA_NAO_PERIGO.glob("*.wav")))
print(f"✅ {gerados} arquivos gerados!")
print(f"   nao_perigo agora: {total_final} arquivos")
print(f"   perigo:           {total_perigo} arquivos")


# ✅ CÉLULA 2 — Extrair features
# Cole numa célula nova e rode DEPOIS

import librosa
import numpy as np
from pathlib import Path
from tqdm import tqdm
import pickle

DATASET_DIR = Path("dataset_binario")
FEATURES_FILE = "features_final.pkl"
N_MFCC = 40
MAX_PAD = 174

def extrair_mfcc(caminho_audio):
    try:
        audio, sr = librosa.load(str(caminho_audio), sr=22050, duration=5.0)
        mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=N_MFCC)
        if mfcc.shape[1] < MAX_PAD:
            mfcc = np.pad(mfcc, ((0, 0), (0, MAX_PAD - mfcc.shape[1])), mode="constant")
        else:
            mfcc = mfcc[:, :MAX_PAD]
        return mfcc.flatten()
    except:
        return None

print("🔍 Extraindo features...\n")
X, y = [], []
classes = sorted([p.name for p in DATASET_DIR.iterdir() if p.is_dir()])
class_map = {nome: idx for idx, nome in enumerate(classes)}
print(f"Classes: {classes}\n")

for classe in classes:
    pasta = DATASET_DIR / classe
    arquivos = list(pasta.glob("*.wav")) + list(pasta.glob("*.mp3"))
    print(f"📁 {classe} — {len(arquivos)} arquivos")
    for arquivo in tqdm(arquivos, desc=f"   {classe}"):
        features = extrair_mfcc(arquivo)
        if features is not None:
            X.append(features)
            y.append(class_map[classe])

X = np.array(X)
y = np.array(y)

with open(FEATURES_FILE, "wb") as f:
    pickle.dump({"X": X, "y": y, "classes": classes}, f)

print(f"\n✅ Features extraídas! Shape: {X.shape}")


# ✅ CÉLULA 3 — Treinar modelo final equilibrado
# Cole numa célula nova e rode DEPOIS

import pickle
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, confusion_matrix
import joblib

with open("features_final.pkl", "rb") as f:
    data = pickle.load(f)

X, y, classes = data["X"], data["y"], data["classes"]
print(f"📦 Dataset: {X.shape[0]} amostras | Classes: {classes}\n")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

print("🧠 Treinando modelo final equilibrado...\n")
modelo = MLPClassifier(
    hidden_layer_sizes=(256, 128, 64),
    activation="relu",
    max_iter=300,
    random_state=42,
    verbose=True,
    early_stopping=True,
    validation_fraction=0.1,
    # Compensa qualquer diferença residual entre classes
    # (não há parâmetro class_weight direto no MLP,
    #  mas o balanceamento foi feito nos dados)
)
modelo.fit(X_train, y_train)

print("\n📊 AVALIAÇÃO MODELO FINAL EQUILIBRADO")
print("=" * 50)
y_pred = modelo.predict(X_test)
print(classification_report(y_test, y_pred, target_names=classes))

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=classes, yticklabels=classes)
plt.title("Matriz de Confusão — Modelo Final Equilibrado")
plt.ylabel("Real")
plt.xlabel("Previsto")
plt.tight_layout()
plt.savefig("matriz_confusao_final_equilibrado.png", dpi=150)
plt.show()

joblib.dump(modelo, "modelo_final_equilibrado.pkl")
joblib.dump(scaler, "scaler_final_equilibrado.pkl")
print("\n💾 modelo_final_equilibrado.pkl")
print("💾 scaler_final_equilibrado.pkl")

In [ ]:
# ✅ TESTE — Modelo Final Equilibrado com upload de áudio
# Cole numa célula nova e rode

import librosa
import numpy as np
import joblib
import pickle
from google.colab import files

# Carregar modelo final
modelo = joblib.load("modelo_final_equilibrado.pkl")
scaler = joblib.load("scaler_final_equilibrado.pkl")
with open("features_final.pkl", "rb") as f:
    classes = pickle.load(f)["classes"]

print("✅ Modelo carregado!")
print(f"   Classes: {classes}\n")

def prever_audio(caminho):
    try:
        audio, sr = librosa.load(caminho, sr=22050, duration=5.0)
        mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=40)
        if mfcc.shape[1] < 174:
            mfcc = np.pad(mfcc, ((0,0),(0, 174 - mfcc.shape[1])), mode="constant")
        else:
            mfcc = mfcc[:, :174]
        features = scaler.transform([mfcc.flatten()])
        pred = modelo.predict(features)[0]
        proba = modelo.predict_proba(features)[0]
        classe = classes[pred]
        confianca = proba[pred]

        emoji = "🚨" if classe == "perigo" else "✅"
        print(f"{emoji} Resultado: {classe.upper()} ({confianca:.1%} de confiança)")
        print("\nProbabilidades:")
        for c, p in zip(classes, proba):
            barra = "█" * int(p * 40)
            print(f"  {c:<15} {p:.1%}  {barra}")
    except Exception as e:
        print(f"[ERRO] {e}")

# Upload de áudio
print("📂 Selecione um arquivo de áudio (.wav, .mp3 ou .m4a):")
uploaded = files.upload()

for nome in uploaded.keys():
    print(f"\n🔊 Analisando: {nome}")
    prever_audio(nome)
    print("-" * 45)